In [0]:

CREATE OR REPLACE TABLE steam_game_intelligence.silver.steam_games
USING DELTA
AS
WITH ranked_snapshots AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY app_id
            ORDER BY
                _ingested_at DESC NULLS LAST,
                CASE
                    WHEN name IS NOT NULL AND TRIM(name) <> ''
                    THEN 1 ELSE 0
                END DESC,
                CASE WHEN price IS NOT NULL THEN 1 ELSE 0 END DESC,
                CASE WHEN release_date IS NOT NULL THEN 1 ELSE 0 END DESC,
                CASE
                    WHEN developers IS NOT NULL
                         AND TRIM(developers) <> ''
                    THEN 1 ELSE 0
                END DESC,
                CASE
                    WHEN genres IS NOT NULL AND TRIM(genres) <> ''
                    THEN 1 ELSE 0
                END DESC,
                CASE
                    WHEN categories IS NOT NULL
                         AND TRIM(categories) <> ''
                    THEN 1 ELSE 0
                END DESC,
                CASE WHEN positive IS NOT NULL THEN 1 ELSE 0 END DESC,
                CASE WHEN negative IS NOT NULL THEN 1 ELSE 0 END DESC
        ) AS snapshot_rank
    FROM steam_game_intelligence.bronze.steam_games
    WHERE app_id IS NOT NULL
      AND app_id > 0
),
latest_games AS (
    SELECT *
    FROM ranked_snapshots
    WHERE snapshot_rank = 1
      AND name IS NOT NULL
      AND TRIM(name) <> ''
)
SELECT
    app_id,
    TRIM(name) AS name,
    release_date,
    price,
    NULLIF(TRIM(price_status), '') AS price_status,
    NULLIF(TRIM(estimated_owners), '') AS estimated_owners,
    NULLIF(TRIM(developers), '') AS developers,
    NULLIF(TRIM(publishers), '') AS publishers,
    NULLIF(TRIM(genres), '') AS genres,
    NULLIF(TRIM(categories), '') AS categories,
    NULLIF(TRIM(tags), '') AS tags,
    positive,
    negative,
    recommendations,
    peak_ccu,
    metacritic_score,
    user_score,
    average_playtime_forever,
    median_playtime_forever,
    average_playtime_2weeks,
    median_playtime_2weeks,
    NULLIF(TRIM(short_description), '') AS short_description,
    NULLIF(TRIM(about_the_game), '') AS about_the_game,
    NULLIF(TRIM(detailed_description), '') AS detailed_description,
    NULLIF(TRIM(notes), '') AS notes,
    achievements,
    dlc_count,
    NULLIF(TRIM(packages), '') AS packages,
    NULLIF(TRIM(supported_languages), '') AS supported_languages,
    NULLIF(TRIM(full_audio_languages), '') AS full_audio_languages,
    windows,
    mac,
    linux,
    NULLIF(TRIM(header_image), '') AS header_image,
    NULLIF(TRIM(screenshots), '') AS screenshots,
    NULLIF(TRIM(movies), '') AS movies,
    NULLIF(TRIM(website), '') AS website,
    NULLIF(TRIM(support_url), '') AS support_url,
    NULLIF(TRIM(support_email), '') AS support_email,
    NULLIF(TRIM(metacritic_url), '') AS metacritic_url,
    steam_store_available,
    steam_spy_available,
    _ingested_at,
    NULLIF(TRIM(_source), '') AS _source
FROM latest_games;


In [0]:

SELECT
    COUNT(*) AS silver_rows,
    COUNT(DISTINCT app_id) AS distinct_app_ids,
    COUNT_IF(app_id IS NULL OR app_id <= 0) AS invalid_app_ids,
    COUNT_IF(name IS NULL OR TRIM(name) = '') AS missing_names,
    COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamps,
    MIN(_ingested_at) AS oldest_selected_snapshot,
    MAX(_ingested_at) AS newest_selected_snapshot
FROM steam_game_intelligence.silver.steam_games;


In [0]:

SELECT
    app_id,
    COUNT(*) AS row_count
FROM steam_game_intelligence.silver.steam_games
GROUP BY app_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC;


In [0]:
SELECT
    'bronze' AS layer,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT app_id) AS distinct_app_ids
FROM steam_game_intelligence.bronze.steam_games

UNION ALL

SELECT
    'silver' AS layer,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT app_id) AS distinct_app_ids
FROM steam_game_intelligence.silver.steam_games;